# Init

In [1]:
#Imports
import os
import sys
import numpy as np
import yaml
import shutil
# add root directory to be able to import packages
# todo: make all packages installable so they can be called/imported by environment
module_path = os.path.abspath(os.path.join('../'))
sys.path.append(module_path)

%matplotlib inline
#%matplotlib tk
%autosave 180
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

from Classes import *
from Helper import *

class information_extractor:
    def __init__(self, root_dir, save_dir="extracted", wanted_animal_ids=["all"], wanted_session_ids=["all"], print_loading=True):
        Animal.root_dir = root_dir #TODO: could make problems if using multiple extractors
        self.root_dir = root_dir
        self.save_path = os.path.join(root_dir, save_dir)
        dir_exist_create(self.save_path)
        self.animals = load_all(root_dir, wanted_animal_ids=wanted_animal_ids,
                            wanted_session_ids=wanted_session_ids, 
                            print_loading=print_loading)

    def from_cabincorr(self, unit_id, data_types, animals=None):
        data_types = make_list_ifnot(data_types)
        animals = self.animals if not animals else animals
        for animal_id, session_id, session in yield_animal_session(animals):
            bin_traces_zip = session.load_cabincorr_data(unit_id=unit_id)
            if not bin_traces_zip:
                print(f"No CaBincorrPath found for unit {unit_id} in {session.session_dir}")
                continue
            for data_type in data_types:
                data = bin_traces_zip[data_type]
                create_dirs([self.save_path, animal_id, session_id])
                save_file_path = os.path.join(self.save_path, animal_id, session_id, data_type+".npy")
                if not save_file_present(save_file_path):
                    np.save(save_file_path, data)

    def cabincorr(self, folder_name_content="merged", animals=None):
        animals = self.animals if not animals else animals
        for animal_id, session_id, session in yield_animal_session(animals):
            for cabincorr_data_path in session.cabincorr_data_paths:
                if folder_name_content in cabincorr_data_path:
                    if os.path.exists(session.cabincorr_fpath):
                        create_dirs([self.save_path, animal_id, session_id])
                        new_path = create_dirs([self.save_path, session.animal_id, session.session_id])
                        save_file_path = os.path.join(new_path, Session.cabincorr_fname)
                        if not save_file_present(save_file_path):
                            shutil.copy(session.cabincorr_fpath, new_path)
                    else:
                        print(f"No {Session.cabincorr_fname} found in {session.cabincorr_fpath}")

    def yaml(self, animals=None): 
        animals = self.animals if not animals else animals
        for animal_id, animal in animals.items():
            create_dirs([self.save_path, animal_id])
            yaml_fname = animal.animal_id+".yaml"
            old_path = os.path.join(animal.animal_dir, yaml_fname)
            new_path = create_dirs([self.save_path, animal_id])
            new_file_path = os.path.join(new_path, yaml_fname)
            if not save_file_present(new_file_path):
                shutil.copy(old_path, new_path)

    def from_suite2p_folder(self, file_names, folder_name_content="merged", animals=None):
        animals = self.animals if not animals else animals
        for file_name in file_names:
            for animal_id, session_id, session in yield_animal_session(animals):
                for s2p_path in session.suite2p_paths:
                    if folder_name_content in s2p_path:
                        fpath = search_file(s2p_path, file_name)
                        if fpath:
                            save_path = create_dirs([self.save_path, animal_id, session_id])
                            save_file_path = os.path.join(save_path, file_name)
                            if not save_file_present(save_file_path):
                                shutil.copy(fpath, save_path)

Autosaving every 180 seconds


# Usage

In [ ]:
root_dir = "/scicore/projects/donafl00-calcium/Users/Sergej/Steffen_Experiments" 
mice21 = ["DON-002865", "DON-003165", "DON-003343", "DON-006084", "DON-006085", "DON-006087"]
mice22 = ["DON-008497", "DON-008498", "DON-008499", "DON-009191", "DON-009192", "DON-010473", "DON-010477"]
mice23 = ["DON-014837", "DON-014838", "DON-014840", "DON-014847", "DON-014849", "DON-015078", "DON-015079"]
wanted_animal_ids = mice21+mice22
#ex = information_extractor(root_dir, wanted_animal_ids=wanted_animal_ids, print_loading=False)

: 

In [ ]:
ex_cecillia = information_extractor(root_dir, save_dir="cecillia", wanted_animal_ids=wanted_animal_ids, print_loading=False)
#ex.cabincorr(folder_name_content="merged")
ex_cecillia.from_cabincorr(unit_id="merged", data_types=["F_detrended", "F_upphase"])
ex_cecillia.yaml()

: 

In [ ]:
ex_rodrigo = information_extractor(root_dir, save_dir="rodrigo", wanted_animal_ids=wanted_animal_ids, print_loading=False)
ex_rodrigo.from_suite2p_folder(file_names=["allcell_clean_corr_pval_zscore.npy"])
ex_rodrigo.yaml()